In [ ]:
"""eod_sale_service — Silver build. Conform the 3 service collections to one line-grain
table (transaction x item): parse nested items/orderInfo, explode items, decode customer
demographics, rename camelCase -> snake_case. All three share ONE doc shape; they differ
only in which fields are populated (a superset schema parses absent fields to null).
"""
from pyspark.sql import functions as F

from ssv_data.io.writers import write_delta
from ssv_data.schema.registry import SERVICE_ITEM_SCHEMA, SERVICE_ORDER_INFO_SCHEMA
from ssv_data.transforms.demographics import decode_customer_demographics

SERVICE_COLLECTIONS = ["pay_bill_transaction", "pay_card_transaction", "top_up_transaction"]


In [ ]:
def _parse_json(df, col, schema):
    """Parse a nested column regardless of how bronze stored it (JSON string from a
    Copy/JDBC source, or struct/array from the Mongo Spark connector)."""
    dt = dict(df.dtypes).get(col, "string")
    src = F.col(col) if dt.startswith("string") else F.to_json(F.col(col))
    return F.from_json(src, schema)


def _conform_service(ctx, table):
    df = ctx.spark.read.table(table)
    df = (df
          .withColumn("oi", _parse_json(df, "orderInfo", SERVICE_ORDER_INFO_SCHEMA))
          .withColumn("item", F.explode_outer(_parse_json(df, "items", SERVICE_ITEM_SCHEMA))))
    it, oi = "item.", "oi."
    return df.select(
        F.col("transactionId").alias("transaction_id"),
        F.col("transactionTime").alias("transaction_time"),
        F.col("postingTime").alias("posting_time"),
        F.col("storeId").alias("store_id"),
        F.col("storeName").alias("store_name"),
        F.col("storeType").alias("store_type"),
        F.col("movementType").alias("movement_type"),
        F.col("hasInvoice").alias("has_invoice"),
        F.col("customerAgeRange").alias("customer_age_range"),
        F.col("customerNationality").alias("customer_nationality"),
        F.col("customerGender").alias("customer_gender"),
        F.col("edcType").alias("edc_type"),
        F.col("cash").alias("cash"),
        F.col("serviceName").alias("service_name"),
        F.col("amountBeforeVat").alias("amount_before_vat"),
        F.col("amountVat").alias("amount_vat"),
        F.col("totalAmountWithoutPromotion").alias("total_amount_without_promotion"),
        F.col("staffId").alias("staff_id"),
        F.col("staffName").alias("staff_name"),
        F.col("posId").alias("pos_id"),
        F.col("paymentMethod").alias("payment_method"),
        F.col("paymentMethodValue").alias("payment_method_value"),
        F.col("transactionRecordStartTime").alias("transaction_record_start_time"),
        F.col("transactionRecordEndTime").alias("transaction_record_end_time"),
        F.col("promotionTotalAmount").alias("promotion_total_amount"),
        F.col("totalProductPromotionAmountWithoutTax").alias("total_product_promotion_amount_without_tax"),
        F.col("totalVoucherPromotionAmountWithoutTax").alias("total_voucher_promotion_amount_without_tax"),
        F.col("totalBillPromotionAmountWithoutTax").alias("total_bill_promotion_amount_without_tax"),
        F.col("serviceFee").alias("service_fee"),
        F.col("customerServiceFee").alias("customer_service_fee"),
        F.col("skipValidateOver50Percent").alias("skip_validate_over_50_percent"),
        F.col("receivableAmount").alias("receivable_amount"),
        F.col("prePaid").alias("pre_paid"),
        F.col("syncedSapStatus").alias("synced_sap_status"),
        F.col("syncedSapTime").alias("synced_sap_time"),
        F.col(oi + "orderNo").alias("order_no"),
        F.col(oi + "serviceId").alias("service_id"),
        F.col(oi + "providerId").alias("provider_id"),
        F.col(oi + "customerId").alias("customer_id"),
        F.col(oi + "orderType").alias("order_type"),
        F.col(it + "productCode").alias("product_code"),
        F.col(it + "productName").alias("product_name"),
        F.col(it + "productUomId").alias("product_uom_id"),
        F.col(it + "supplierCode").alias("supplier_code"),
        F.col(it + "supplierName").alias("supplier_name"),
        F.col(it + "quantity").alias("quantity"),
        F.col(it + "purchasePriceWithTax").alias("purchase_price_with_tax"),
        F.col(it + "purchasePriceWithoutTax").alias("purchase_price_without_tax"),
        F.col(it + "totalAmount").alias("total_amount"),
        F.col(it + "retailBusinessType").alias("retail_business_type"),
        F.col(it + "commissionOnVnd").alias("commission_on_vnd"),
        F.col(it + "totalCommission").alias("total_commission"),
    )


In [ ]:
def build_service_line(ctx):
    df = None
    for coll in SERVICE_COLLECTIONS:
        tbl = ctx.bronze(coll)
        if not ctx.spark.catalog.tableExists(tbl):
            ctx.logger.info(f"silver(service): {tbl} absent — skipped")
            continue
        c = _conform_service(ctx, tbl)
        df = c if df is None else df.unionByName(c)
    if df is None:
        raise ValueError("silver(service): no bronze service collections found")
    # Service legacy PASSES THROUGH unmapped age codes (e.g. '0'); gender/nationality decode.
    df = decode_customer_demographics(df, keep_unknown_age=True)
    return (df
            .withColumn("skip_validate_over_50_percent_by_int",
                        F.col("skip_validate_over_50_percent").cast("int"))
            .withColumn("report_date", F.to_date("transaction_time")))


def service_silver_build(ctx) -> None:
    line = build_service_line(ctx)
    # Keep only the run-day partition so the write matches replaceWhere exactly.
    line = line.where(F.col("report_date") == F.to_date(F.lit(ctx.running_date)))
    write_delta(ctx, line, ctx.silver("sale_service_line"),
                partition_col="report_date", replace_where=ctx.replace_where)
    ctx.logger.info("silver(service) build complete")
